# Keyness: Which Words Distinguish a Corpus

**Computational Text Analysis Pack · Module 04 — Comparing Corpora**

*Technique:* Log-likelihood keyness; target vs reference corpus

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/04a_keyness.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Frequency alone told you the commonest words are *ni*, *ti* and *the*. Useless.

**Keyness** asks a better question: which words are more frequent *here* than you
would expect, given how common they are overall? That needs two corpora — a
**target** (what you are studying) and a **reference** (what you are comparing it
against).

The comparison is the method. Change the reference corpus and you change every
answer — which is why the reference belongs in your methods section, named.

## The tooling

**`collections.Counter`** and plain arithmetic. Keyness needs no library: the
log-likelihood formula is four lines. **quanteda** (R) and **Sketch Engine** are
the field's usual implementations of exactly this.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
def log_likelihood(counts_a, counts_b, min_count=10):
    """Words that distinguish corpus A from corpus B.

    The G-squared statistic corpus linguists call *keyness*. A word scores high
    when it is far more common in one corpus than its overall rate predicts.
    Positive `diff` means over-used in A.
    """
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    rows = []
    for w in set(counts_a) | set(counts_b):
        o1, o2 = counts_a.get(w, 0), counts_b.get(w, 0)
        if o1 + o2 < min_count:
            continue
        e1 = na * (o1 + o2) / (na + nb)
        e2 = nb * (o1 + o2) / (na + nb)
        g2 = 2 * ((o1 * np.log(o1 / e1) if o1 else 0)
                  + (o2 * np.log(o2 / e2) if o2 else 0))
        rows.append({"word": w, "count_a": o1, "count_b": o2,
                     "per_1000_a": o1 / na * 1000, "per_1000_b": o2 / nb * 1000,
                     "keyness": g2, "over_used_in": "A" if o1 / na > o2 / nb else "B"})
    out = pd.DataFrame(rows).sort_values("keyness", ascending=False)
    return out.reset_index(drop=True)

## Target and reference

Start with something whose answer you can check: health reporting against all
other reporting, in one language. If the method works, *health* vocabulary comes
out on top.

In [ ]:
LANGUAGE = "English"        # ← change this
TARGET_CATEGORY = "health"  # ← and this

sub = df[df["language"] == LANGUAGE]
target = sub[sub["category"] == TARGET_CATEGORY]
reference = sub[sub["category"] != TARGET_CATEGORY]

print(f"target    : {len(target):>5} {TARGET_CATEGORY} documents")
print(f"reference : {len(reference):>5} documents in every other category")

counts_t = frequencies(target["text"])
counts_r = frequencies(reference["text"])
keys = log_likelihood(counts_t, counts_r, min_count=10)

over = keys[keys["over_used_in"] == "A"].head(20)
print()
print(over[["word", "count_a", "per_1000_a", "per_1000_b", "keyness"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Read it as a corpus linguist would

Three columns matter together:

- **keyness** — how confident you are the difference is not chance
- **per_1000_a vs per_1000_b** — how *big* the difference is
- **count_a** — how much evidence it rests on

A high keyness on a word occurring 11 times is statistically real and
substantively nothing. Report all three, never keyness alone.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
top = over.head(15).iloc[::-1]
ax.barh(top["word"], top["keyness"])
ax.set_xlabel("keyness (log-likelihood G²)")
ax.set_title(f"Words distinctive of {TARGET_CATEGORY} reporting ({LANGUAGE})")
plt.tight_layout(); plt.show()

In [ ]:
# Under-used words: what this corpus talks about LESS than the reference.
# Usually ignored, often more interesting than the over-used list.
under = keys[keys["over_used_in"] == "B"].head(15)
print(f"Words markedly ABSENT from {TARGET_CATEGORY} reporting:")
print(under[["word", "per_1000_a", "per_1000_b", "keyness"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

### 🔧 Your turn 1

Change `TARGET_CATEGORY` to `"politics"`, then `"sports"`, and re-run.

Do the results look like what you know about those subjects? If they do, the
method is working and you can trust it on a question you do not already know the
answer to. That check is not optional.

## Changing the reference changes everything

The same target corpus, compared against two different references, gives two
different answers. Neither is wrong; they answer different questions.

In [ ]:
hau = df[df["language"] == "Hausa"]
c_hau = frequencies(hau["text"])

for ref_name, ref_df in [("all other African-language news",
                          df[df["language"] != "Hausa"]),
                         ("Nigerian Pidgin news only",
                          df[df["language"] == "Nigerian Pidgin"])]:
    c_ref = frequencies(ref_df["text"])
    k = log_likelihood(c_hau, c_ref, min_count=20)
    words = ", ".join(k[k["over_used_in"] == "A"].head(10)["word"])
    print(f"Hausa vs {ref_name}:")
    print(f"   {words}")
    print()
print("Against a multilingual reference you mostly recover 'this is Hausa'.")
print("Against a single other language you recover something closer to a")
print("contrast of subject matter. Choose the reference to fit the question.")

### 🔧 Your turn 2

Pick a target and reference that answer a question you actually care about — one
category against another, or one language's coverage of a topic against another's.

Write the pair down as a sentence: *"I compared X against Y in order to find out
Z."* If you cannot finish that sentence, the comparison is not yet a question.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Health reporting should surface disease, treatment and health-
system vocabulary; politics should surface party, election and office terms;
sports should surface team and match terms. If a category returns something
unexpected, look at the *reference*: a category with few documents borrows its
apparent distinctiveness from whatever dominates the rest of the corpus.

**Your turn 2.** The sentence is the test. "I compared Hausa health reporting
against Hausa political reporting to find out whether health coverage uses more
first-person testimony" is a question. "I compared Hausa to English" is a
comparison with no question attached — it will produce a list of Hausa words, and
you will have learned that Hausa is Hausa.

</details>

## Where this stops

You can find the words that distinguish one corpus from another. What you cannot
yet do is say how *large* the difference is — keyness confounds size of effect
with size of corpus. That is 04b.